In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#Data Overview

## NIST Photovoltaic Dataset

Dataset Source:  
https://pvdata.nist.gov/

---

## General Information

| Attribute | Description |
|---|---|
| **Location** | Maryland, United States |
| **Data Type** | Real-world photovoltaic system measurements |
| **Time Granularity** | 1-minute intervals |
| **Years Used** | 2015 until 2018 |

---

## Measurement Locations

| Location | Description |
|---|---|
| **Ground** | Solar panels installed on ground-mounted structures (fixed tilt ≈ 20°) |
| **Roof** | Solar panels installed on building rooftops |
| **Canopy** | Solar panels installed as parking canopy/shading structures |
| **ws_1, ws_2** | Independent weather stations collecting meteorological measurements |

---

## Dataset Size

| Dataset | Years Included | Total Rows |
|---|---|---|
| **Canopy Data** | 2015, 2016, 2017, 2018 | 2,102,400 |
| **Ground Data** | 2015, 2016, 2017, 2018 | 2,102,400 |
| **Roof Data** | 2015, 2016, 2017, 2018 | 1,706,400 |

## Dataset Structure Verification and File Counting

The dataset directory structure is scanned to verify that all expected CSV files are present for each month and year.

The dataset is organized hierarchically as:

```text
NIST/
│
├── onemin-Canopy-2015/
│   ├── 01/
│   ├── 02/
│   └── ...
│
├── onemin-Canopy-2016/
│   ├── 01/
│   ├── 02/
│   └── ...
```

Each monthly folder contains daily CSV files, where:

- **1 CSV file = 1 day of measurements**
- Therefore, the number of files in a month should normally match the number of days in that month:
  - 31 files for months with 31 days
  - 30 files for months with 30 days
  - 28 or 29 files for February

The code iterates through all dataset folders and monthly subfolders, counts the number of `.csv` files, and prints the results. This acts as a preliminary data integrity check before preprocessing and model development.

This verification step is important because it helps identify:

- Missing files
- Incomplete downloads
- Corrupted or partially collected data
- Irregularities in the dataset structure

For example:

```text
onemin-Roof-2018 | Month 03 → 18 files
```

indicates that March 2018 contains fewer files than expected, suggesting incomplete data for that period.

The file counts are also stored in a nested Python dictionary (`file_counts`) for later analysis if needed.

In [ ]:
import os

base_path = "/content/drive/MyDrive/NIST"

file_counts = {}

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        file_counts[folder] = {}

        for month in sorted(os.listdir(folder_path)):
            month_path = os.path.join(folder_path, month)

            if os.path.isdir(month_path):
                csv_files = [f for f in os.listdir(month_path) if f.endswith(".csv")]
                count = len(csv_files)

                file_counts[folder][month] = count

                print(f"{folder} | Month {month} → {count} files")


onemin-Canopy-2015 | Month 01 → 31 files
onemin-Canopy-2015 | Month 02 → 28 files
onemin-Canopy-2015 | Month 03 → 31 files
onemin-Canopy-2015 | Month 04 → 30 files
onemin-Canopy-2015 | Month 05 → 31 files
onemin-Canopy-2015 | Month 06 → 30 files
onemin-Canopy-2015 | Month 07 → 31 files
onemin-Canopy-2015 | Month 08 → 31 files
onemin-Canopy-2015 | Month 09 → 30 files
onemin-Canopy-2015 | Month 10 → 31 files
onemin-Canopy-2015 | Month 11 → 30 files
onemin-Canopy-2015 | Month 12 → 31 files
onemin-Canopy-2016 | Month 01 → 31 files
onemin-Canopy-2016 | Month 02 → 29 files
onemin-Canopy-2016 | Month 03 → 31 files
onemin-Canopy-2016 | Month 04 → 30 files
onemin-Canopy-2016 | Month 05 → 31 files
onemin-Canopy-2016 | Month 06 → 30 files
onemin-Canopy-2016 | Month 07 → 31 files
onemin-Canopy-2016 | Month 08 → 31 files
onemin-Canopy-2016 | Month 09 → 30 files
onemin-Canopy-2016 | Month 10 → 31 files
onemin-Canopy-2016 | Month 11 → 30 files
onemin-Canopy-2016 | Month 12 → 31 files
onemin-Canopy-20

## Merging Daily CSV Files

This step combines all daily CSV files for each dataset folder into a single unified CSV file.

The code:

- Iterates through all installation/year folders
- Reads every daily `.csv` file from each monthly subfolder
- Concatenates them into one DataFrame
- Saves the merged dataset into:
  ```text
  /content/drive/MyDrive/NIST_processed
  ```

This preprocessing step simplifies later analysis and machine learning workflows by converting thousands of small daily files into a smaller number of continuous datasets.

The output also helps verify dataset completeness. Since the data has a 1-minute granularity:

- A normal year should contain:
  ```text
  24 × 60 × 365 = 525,600 records
  ```
- A leap year should contain:
  ```text
  24 × 60 × 366 = 527,040 records
  ```

The Canopy datasets match these expected values for 2015–2017, while the 2018 dataset contains fewer records, indicating missing data. Similar inconsistencies are also observed in some Ground and Roof datasets.


Memory is also managed efficiently by saving each merged dataset immediately and deleting it from memory after processing.

In [ ]:
import os

base_path = '/content/drive/MyDrive/NIST'
output_path = '/content/drive/MyDrive/NIST_processed'

os.makedirs(output_path, exist_ok=True)

for folder in sorted(os.listdir(base_path)):
    folder_path = os.path.join(base_path, folder)

    if os.path.isdir(folder_path):
        combined_df = None

        for month in sorted(os.listdir(folder_path)):
            month_path = os.path.join(folder_path, month)

            if os.path.isdir(month_path):
                for file in sorted(os.listdir(month_path)):
                    if file.endswith(".csv"):
                        file_path = os.path.join(month_path, file)

                        df = pd.read_csv(file_path)

                        # Incremental concat (memory safer)
                        if combined_df is None:
                            combined_df = df
                        else:
                            combined_df = pd.concat([combined_df, df], ignore_index=True)

        # Save to disk immediately
        save_path = os.path.join(output_path, f"{folder}.csv")
        combined_df.to_csv(save_path, index=False)

        print(f"Saved: {folder} → {combined_df.shape}")

        # 🔥 CRITICAL: free memory
        del combined_df

Saved: onemin-Canopy-2015 → (525600, 102)
Saved: onemin-Canopy-2016 → (527040, 102)
Saved: onemin-Canopy-2017 → (525600, 102)
Saved: onemin-Canopy-2018 → (525107, 102)
Saved: onemin-Ground-2015 → (525588, 100)
Saved: onemin-Ground-2016 → (527040, 100)
Saved: onemin-Ground-2017 → (525595, 100)
Saved: onemin-Ground-2018 → (525587, 100)
Saved: onemin-Roof-2015 → (523450, 123)
Saved: onemin-Roof-2016 → (521270, 123)
Saved: onemin-Roof-2017 → (525599, 123)
Saved: onemin-Roof-2018 → (110838, 123)


#Data Inspection

In [ ]:
canopy_2015 = pd.read_csv('/content/drive/MyDrive/NIST_processed/onemin-Canopy-2015.csv')
canopy_2016 = pd.read_csv('/content/drive/MyDrive/NIST_processed/onemin-Canopy-2016.csv')

In [ ]:
canopy_2015

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RECORD,CR1000Temp_C_Avg,DoorOpen_Min,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,...,RTD_C_Avg_1,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10
0,2015-01-01 00:00:00-05:00,-4.433096,-8.782201,-4.542599,173899.0,-3.470,0.0,-0.548,-0.040,-2.229,...,-8.620,-8.290,-8.680,-8.480,-8.310,-8.660,-8.67,-8.100,-7.542,1.635
1,2015-01-01 00:01:00-05:00,-4.433096,-8.782201,-4.542599,173900.0,-3.476,0.0,-0.548,-0.040,-2.099,...,-8.610,-8.350,-8.520,-8.180,-8.020,-8.470,-8.40,-7.988,-7.385,1.635
2,2015-01-01 00:02:00-05:00,-4.433096,-8.782201,-4.542599,173901.0,-3.493,0.0,-0.550,-0.041,-2.185,...,-8.630,-8.390,-8.470,-8.140,-7.813,-8.380,-8.27,-7.973,-7.366,1.618
3,2015-01-01 00:03:00-05:00,-4.433096,-8.977362,-4.542599,173902.0,-3.493,0.0,-0.547,-0.038,-2.261,...,-8.610,-8.450,-8.450,-8.100,-7.831,-8.380,-8.25,-7.941,-7.367,1.618
4,2015-01-01 00:04:00-05:00,-4.669528,-9.270102,-4.542599,173903.0,-3.507,0.0,-0.545,-0.036,-2.391,...,-8.560,-8.450,-8.550,-8.070,-7.799,-8.380,-8.25,-7.973,-7.449,1.539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525595,2015-12-31 23:55:00-05:00,-3.605585,-7.220922,-3.420970,275781.0,6.701,0.0,-0.305,-0.051,5.109,...,3.836,3.822,3.646,3.998,3.759,3.845,-138.90,4.077,4.127,9.010
525596,2015-12-31 23:56:00-05:00,-3.605585,-7.123341,-3.420970,275782.0,6.701,0.0,-0.305,-0.051,5.080,...,3.836,3.808,3.646,3.966,3.742,3.813,-139.10,4.028,4.127,8.980
525597,2015-12-31 23:57:00-05:00,-3.605585,-7.123341,-3.420970,275783.0,6.685,0.0,-0.305,-0.051,5.062,...,3.836,3.740,3.628,3.934,3.742,3.749,-138.90,3.999,4.127,8.950
525598,2015-12-31 23:58:00-05:00,-3.605585,-7.123341,-3.420970,275784.0,6.678,0.0,-0.305,-0.051,5.081,...,3.836,3.740,3.550,3.934,3.727,3.749,-139.00,3.981,4.110,8.960


In [ ]:
canopy_2015.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525600 entries, 0 to 525599
Columns: 102 entries, TIMESTAMP to RTD_C_Avg_10
dtypes: float64(101), object(1)
memory usage: 409.0+ MB


In [ ]:
canopy_2015.describe()

,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RECORD,CR1000Temp_C_Avg,DoorOpen_Min,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,...,RTD_C_Avg_1,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10
count,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,...,525235.000000,525235.000000,525235.000000,525235.000000,525235.000000,525234.000000,525235.000000,525235.000000,525235.000000,525235.000000
mean,167.429285,160.167997,168.245299,205021.662555,17.473426,0.026702,161.326398,159.277743,13.744469,192.028275,...,16.697201,17.016565,16.370072,16.671378,16.130982,16.680692,-27.761905,16.771227,16.252387,23.998807
std,270.211489,268.332786,271.639032,110432.411666,12.386112,2.609290,256.743640,254.657250,10.628394,166.904965,...,15.398464,15.701015,15.731751,15.521012,14.985293,15.698146,62.395839,15.497054,14.954201,14.479788
min,-12.944641,-20.686964,-9.141280,0.000000,-15.570000,0.000000,-0.581000,-0.254000,-17.010000,0.075000,...,-20.060000,-20.770000,-21.170000,-20.020000,-20.260000,-18.530000,-148.500000,-20.700000,-19.830000,-15.160000
25%,-3.960233,-8.489461,-3.869622,119560.500000,8.360000,0.000000,-0.304000,-0.125000,5.900000,8.590000,...,5.368000,5.562000,4.907000,5.315500,5.192000,5.150000,-98.600000,5.465000,5.368000,13.150000
50%,-0.413756,-4.586261,-0.168245,212384.000000,17.930000,0.000000,3.543000,1.734000,14.830000,308.600000,...,15.760000,15.980000,15.180000,15.590000,15.290000,15.490000,0.328000,15.720000,15.420000,24.480000
75%,253.868640,239.363780,253.993010,280290.500000,25.750000,0.000000,238.700000,234.000000,22.380000,347.000000,...,24.650000,24.970000,24.120000,24.490000,23.990000,24.560000,18.350000,24.590000,24.100000,34.450000
max,1429.171000,1418.813500,1436.751200,411599.000000,48.240000,255.000000,1385.000000,1391.000000,35.980000,469.700000,...,64.300000,67.730000,64.590000,63.870000,61.880000,64.060000,65.680000,64.210000,63.090000,58.280000


In [ ]:
canopy_2015.shape

(525600, 102)

In [ ]:
canopy_2015.size

53611200

In [ ]:
for column in canopy_2015.columns:
  print(column)

TIMESTAMP
Pyra1_Wm2_Avg
Pyra2_Wm2_Avg
Pyra3_Wm2_Avg
RECORD
CR1000Temp_C_Avg
DoorOpen_Min
RefCell1_Wm2_Avg
RefCell2_Wm2_Avg
AmbTemp_C_Avg
InvVDVoltage_V_Avg
InvVa_Avg
InvVb_Avg
InvVc_Avg
InvIa_Avg
InvIb_Avg
InvIc_Avg
InvVDCin_Avg
InvIDCin_Avg
InvFreq_Avg
InvPAC_kW_Avg
InvEtot_kWh_Max
InvVPVin_Avg
InvPDC_kW_Avg
InvOpStatus_Avg
InvMainFault_Max
InvDriveFault_Max
InvVoltageFault_Max
InvGridFault_Max
InvTempFault_Max
InvSystemFault_Max
InvSystemWarn_Max
InvPVMStatus_Max
SEWSAmbientTemp_C_Avg
SEWSModuleTemp_C_Avg
SEWSPOAIrrad_Wm2_Avg
PwrMtrIa_Avg
PwrMtrIb_Avg
PwrMtrIc_Avg
PwrMtrFreq_Avg
PwrMtrFreqMin_Min
PwrMtrFreqMax_Max
PwrMtrPhaseRev_Avg
PwrMtrVa_Avg
PwrMtrVb_Avg
PwrMtrVc_Avg
PwrMtrP_kW_Avg
PwrMtrP_kVAR_Avg
PwrMtrP_kVA_Avg
PwrMtrEdel_kWh_Max
PwrMtrErec_kWh_Max
PwrMtrEdel_kVARh_Max
PwrMtrErec_kVARh_Max
PwrMtrPF_Avg
PwrMtrVaTHDMax_Max
PwrMtrVbTHDMax_Max
PwrMtrVcTHDMax_Max
ShuntCurrent_A_Avg_1
ShuntCurrent_A_Avg_2
ShuntCurrent_A_Avg_3
ShuntCurrent_A_Avg_4
ShuntCurrent_A_Avg_5
ShuntCurrent_A_

In [ ]:
secondary_features = ['RECORD', 'CR1000Temp_C_Avg', 'DoorOpen_Min', 'Battery_V_Min', 'Battery_A_Avg', 'Load_A_Avg', 'ChgState_Min', 'ChgSource_Min', 'CkBatt_Max', 'Qloss_Ah_Max','WindDirAve_deg', 'WindDirStdDev_deg', 'WindSpeed_ms_Max', 'WindRef_V_Min', 'WindHeatStateID_Avg', 'WindValid_Avg', 'PwrMtrP_kVAR_Avg', 'PwrMtrEdel_kVARh_Max', 'PwrMtrErec_kVARh_Max', 'ShuntEtot_kWh_Max_1','ShuntEtot_kWh_Max_2', 'ShuntEtot_kWh_Max_3', 'ShuntEtot_kWh_Max_4', 'ShuntEtot_kWh_Max_5', 'ShuntEtot_kWh_Max_6', 'ShuntEtot_kWh_Max_7']

canopy_2015 = canopy_2015.drop(columns = secondary_features)
canopy_2015

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_1,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10
0,2015-01-01 00:00:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.229,11.75,277.4,276.5,...,-8.620,-8.290,-8.680,-8.480,-8.310,-8.660,-8.67,-8.100,-7.542,1.635
1,2015-01-01 00:01:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.099,11.69,277.4,276.5,...,-8.610,-8.350,-8.520,-8.180,-8.020,-8.470,-8.40,-7.988,-7.385,1.635
2,2015-01-01 00:02:00-05:00,-4.433096,-8.782201,-4.542599,-0.550,-0.041,-2.185,11.62,277.4,276.5,...,-8.630,-8.390,-8.470,-8.140,-7.813,-8.380,-8.27,-7.973,-7.366,1.618
3,2015-01-01 00:03:00-05:00,-4.433096,-8.977362,-4.542599,-0.547,-0.038,-2.261,11.57,277.4,276.5,...,-8.610,-8.450,-8.450,-8.100,-7.831,-8.380,-8.25,-7.941,-7.367,1.618
4,2015-01-01 00:04:00-05:00,-4.669528,-9.270102,-4.542599,-0.545,-0.036,-2.391,11.49,277.3,276.5,...,-8.560,-8.450,-8.550,-8.070,-7.799,-8.380,-8.25,-7.973,-7.449,1.539
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525595,2015-12-31 23:55:00-05:00,-3.605585,-7.220922,-3.420970,-0.305,-0.051,5.109,9.64,275.5,275.1,...,3.836,3.822,3.646,3.998,3.759,3.845,-138.90,4.077,4.127,9.010
525596,2015-12-31 23:56:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.80,275.5,275.2,...,3.836,3.808,3.646,3.966,3.742,3.813,-139.10,4.028,4.127,8.980
525597,2015-12-31 23:57:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.062,9.85,275.6,275.4,...,3.836,3.740,3.628,3.934,3.742,3.749,-138.90,3.999,4.127,8.950
525598,2015-12-31 23:58:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.081,9.86,275.7,275.4,...,3.836,3.740,3.550,3.934,3.727,3.749,-139.00,3.981,4.110,8.960


In [ ]:
similar_columns = ["InvVDVoltage_V_Avg", "InvVa_Avg","InvVb_Avg","InvVc_Avg","InvIa_Avg","InvIb_Avg","InvIc_Avg","InvVDCin_Avg","InvIDCin_Avg","InvFreq_Avg","PwrMtrIa_Avg","PwrMtrIb_Avg","PwrMtrIc_Avg","PwrMtrFreq_Avg","PwrMtrVa_Avg","PwrMtrVb_Avg","PwrMtrVc_Avg","PwrMtrP_kW_Avg"]
similars = canopy_2015[similar_columns]
similars

,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,InvVc_Avg,InvIa_Avg,InvIb_Avg,InvIc_Avg,InvVDCin_Avg,InvIDCin_Avg,InvFreq_Avg,PwrMtrIa_Avg,PwrMtrIb_Avg,PwrMtrIc_Avg,PwrMtrFreq_Avg,PwrMtrVa_Avg,PwrMtrVb_Avg,PwrMtrVc_Avg,PwrMtrP_kW_Avg
0,11.75,277.4,276.5,278.1,0.0,0.0,0.0,20.91,0.0,60.14,0.0,0.0,0.0,60.0,274.0,274.0,276.0,0.0
1,11.69,277.4,276.5,278.2,0.0,0.0,0.0,20.95,0.0,60.13,0.0,0.0,0.0,60.0,274.0,274.0,276.0,0.0
2,11.62,277.4,276.5,278.2,0.0,0.0,0.0,20.99,0.0,60.14,0.0,0.0,0.0,60.0,274.0,274.0,276.0,0.0
3,11.57,277.4,276.5,278.2,0.0,0.0,0.0,20.99,0.0,60.15,0.0,0.0,0.0,60.0,274.0,274.0,276.0,0.0
4,11.49,277.3,276.5,278.2,0.0,0.0,0.0,21.06,0.0,60.14,0.0,0.0,0.0,60.0,274.0,274.0,276.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525595,9.64,275.5,275.1,276.5,0.0,0.0,0.0,21.04,0.0,60.14,0.0,0.0,0.0,60.0,273.0,273.0,274.0,0.0
525596,9.80,275.5,275.2,276.6,0.0,0.0,0.0,20.79,0.0,60.14,0.0,0.0,0.0,60.0,273.0,273.0,274.1,0.0
525597,9.85,275.6,275.4,276.7,0.0,0.0,0.0,21.09,0.0,60.20,0.0,0.0,0.0,60.0,273.0,273.0,274.9,0.0
525598,9.86,275.7,275.4,276.7,0.0,0.0,0.0,21.02,0.0,60.16,0.0,0.0,0.0,60.0,273.0,273.0,275.0,0.0


In [ ]:
similars.corr()

,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,InvVc_Avg,InvIa_Avg,InvIb_Avg,InvIc_Avg,InvVDCin_Avg,InvIDCin_Avg,InvFreq_Avg,PwrMtrIa_Avg,PwrMtrIb_Avg,PwrMtrIc_Avg,PwrMtrFreq_Avg,PwrMtrVa_Avg,PwrMtrVb_Avg,PwrMtrVc_Avg,PwrMtrP_kW_Avg
InvVDVoltage_V_Avg,1.000000,0.004509,0.007386,0.009265,0.482416,0.487367,0.484439,0.940382,0.531138,0.009117,0.562343,0.574224,0.564950,0.002884,-0.274056,-0.259557,-0.226431,0.546499
InvVa_Avg,0.004509,1.000000,0.999933,0.999910,0.506826,0.504376,0.507672,0.260372,0.221702,0.999504,0.012495,0.012514,0.012570,0.000097,0.033088,0.027214,0.035105,0.012515
InvVb_Avg,0.007386,0.999933,1.000000,0.999960,0.510035,0.507597,0.510884,0.263311,0.225508,0.999601,0.016139,0.016180,0.016206,0.000099,0.029780,0.027072,0.033361,0.016150
InvVc_Avg,0.009265,0.999910,0.999960,1.000000,0.511718,0.509291,0.512566,0.265078,0.227544,0.999623,0.018084,0.018146,0.018152,0.000109,0.028180,0.024359,0.033375,0.018074
InvIa_Avg,0.482416,0.506826,0.510035,0.511718,1.000000,0.999952,0.999974,0.608128,0.950056,0.508406,0.867338,0.867378,0.867160,0.003219,-0.218281,-0.196154,-0.194454,0.867848
InvIb_Avg,0.487367,0.504376,0.507597,0.509291,0.999952,1.000000,0.999985,0.612424,0.950637,0.505983,0.868480,0.868655,0.868286,0.003230,-0.218965,-0.196878,-0.194987,0.869124
InvIc_Avg,0.484439,0.507672,0.510884,0.512566,0.999974,0.999985,1.000000,0.610355,0.949533,0.509266,0.866661,0.866789,0.866469,0.003220,-0.218152,-0.196057,-0.194452,0.867290
InvVDCin_Avg,0.940382,0.260372,0.263311,0.265078,0.608128,0.612424,0.610355,1.000000,0.583806,0.264391,0.560725,0.572534,0.563335,0.002142,-0.244555,-0.227719,-0.197196,0.544983
InvIDCin_Avg,0.531138,0.221702,0.225508,0.227544,0.950056,0.950637,0.949533,0.583806,1.000000,0.224002,0.973765,0.973368,0.973602,0.003510,-0.261708,-0.230968,-0.229927,0.973737
InvFreq_Avg,0.009117,0.999504,0.999601,0.999623,0.508406,0.505983,0.509266,0.264391,0.224002,1.000000,0.014048,0.014139,0.014118,0.000125,0.008883,0.005863,0.012975,0.013921


In [ ]:
canopy_2015.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2015_new.csv', index=False)

In [ ]:
for column in canopy_2016.columns:
  print(column)

TIMESTAMP
Pyra1_Wm2_Avg
Pyra2_Wm2_Avg
Pyra3_Wm2_Avg
RECORD
CR1000Temp_C_Avg
DoorOpen_Min
RefCell1_Wm2_Avg
RefCell2_Wm2_Avg
AmbTemp_C_Avg
InvVDVoltage_V_Avg
InvVa_Avg
InvVb_Avg
InvVc_Avg
InvIa_Avg
InvIb_Avg
InvIc_Avg
InvVDCin_Avg
InvIDCin_Avg
InvFreq_Avg
InvPAC_kW_Avg
InvEtot_kWh_Max
InvVPVin_Avg
InvPDC_kW_Avg
InvOpStatus_Avg
InvMainFault_Max
InvDriveFault_Max
InvVoltageFault_Max
InvGridFault_Max
InvTempFault_Max
InvSystemFault_Max
InvSystemWarn_Max
InvPVMStatus_Max
SEWSAmbientTemp_C_Avg
SEWSModuleTemp_C_Avg
SEWSPOAIrrad_Wm2_Avg
PwrMtrIa_Avg
PwrMtrIb_Avg
PwrMtrIc_Avg
PwrMtrFreq_Avg
PwrMtrFreqMin_Min
PwrMtrFreqMax_Max
PwrMtrPhaseRev_Avg
PwrMtrVa_Avg
PwrMtrVb_Avg
PwrMtrVc_Avg
PwrMtrP_kW_Avg
PwrMtrP_kVAR_Avg
PwrMtrP_kVA_Avg
PwrMtrEdel_kWh_Max
PwrMtrErec_kWh_Max
PwrMtrEdel_kVARh_Max
PwrMtrErec_kVARh_Max
PwrMtrPF_Avg
PwrMtrVaTHDMax_Max
PwrMtrVbTHDMax_Max
PwrMtrVcTHDMax_Max
ShuntCurrent_A_Avg_1
ShuntCurrent_A_Avg_2
ShuntCurrent_A_Avg_3
ShuntCurrent_A_Avg_4
ShuntCurrent_A_Avg_5
ShuntCurrent_A_

In [ ]:
secondary_features_2 = ['RECORD', 'CR1000Temp_C_Avg', 'DoorOpen_Min', 'Battery_V_Min', 'Battery_A_Avg', 'Load_A_Avg', 'ChgState_Min', 'ChgSource_Min', 'CkBatt_Max', 'Qloss_Ah_Max','WindDirAve_deg', 'WindDirStdDev_deg', 'WindSpeed_ms_Max', 'WindRef_V_Min', 'WindHeatStateID_Avg', 'WindValid_Avg', 'PwrMtrP_kVAR_Avg', 'PwrMtrEdel_kVARh_Max', 'PwrMtrErec_kVARh_Max', 'ShuntEtot_kWh_Max_1','ShuntEtot_kWh_Max_2', 'ShuntEtot_kWh_Max_3', 'ShuntEtot_kWh_Max_4', 'ShuntEtot_kWh_Max_5', 'ShuntEtot_kWh_Max_6', 'ShuntEtot_kWh_Max_7']

canopy_2016 = canopy_2016.drop(columns = secondary_features_2)
canopy_2016

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_1,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10
0,2016-01-01 00:00:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.089,9.82,275.6,275.6,...,3.740,3.673,3.550,3.838,3.646,3.653,-138.700,3.888,4.095,8.93
1,2016-01-01 00:01:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.095,9.60,275.7,275.6,...,3.723,3.676,3.550,3.838,3.646,3.653,-138.900,3.867,4.067,8.88
2,2016-01-01 00:02:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.098,9.38,275.6,275.6,...,3.726,3.644,3.550,3.870,3.646,3.653,-138.300,3.902,4.127,8.86
3,2016-01-01 00:03:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.086,9.24,276.0,275.4,...,3.740,3.676,3.550,3.934,3.727,3.702,-139.000,3.999,4.127,8.85
4,2016-01-01 00:04:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.12,275.7,275.6,...,3.740,3.676,3.550,3.934,3.742,3.670,-139.200,3.952,4.127,8.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527035,2016-12-31 23:55:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.055,6.882,17.77,276.5,275.1,...,6.141,6.237,5.679,6.045,6.018,5.957,6.005,6.047,6.109,9.22
527036,2016-12-31 23:56:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.055,6.892,18.85,277.1,275.7,...,6.141,6.286,5.646,6.077,6.032,6.004,6.005,6.047,6.094,9.22
527037,2016-12-31 23:57:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.059,6.900,19.34,277.3,276.0,...,6.173,6.318,5.676,6.059,6.029,6.025,6.005,6.047,6.080,9.22
527038,2016-12-31 23:58:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.070,6.901,19.54,277.5,275.9,...,6.237,6.315,5.675,6.045,6.032,6.022,6.005,6.047,6.131,9.22


In [ ]:
canopy_2016.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2016_new.csv', index=False)

#Daylight Mask

In [ ]:
pyra_cols = ['Pyra1_Wm2_Avg', 'Pyra2_Wm2_Avg', 'Pyra3_Wm2_Avg']

median_irradiance = canopy_2015[pyra_cols].median(axis=1)

canopy_2015 = canopy_2015[median_irradiance > 100]
canopy_2015

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_1,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10
503,2015-01-01 08:23:00-05:00,72.761880,100.605000,157.420700,42.12,117.5,-3.053,391.1,276.9,275.7,...,-4.742,-4.678,-7.436,-5.558,-5.723,-4.051,-5.627,-5.018,-5.453,-0.481
504,2015-01-01 08:24:00-05:00,75.717280,103.629980,161.346400,43.98,120.8,-2.963,390.2,276.6,275.8,...,-4.357,-4.293,-7.307,-5.305,-5.484,-4.051,-5.374,-4.743,-5.242,-0.417
505,2015-01-01 08:25:00-05:00,78.199810,106.167070,164.599140,45.75,123.5,-2.893,391.1,276.7,275.9,...,-3.973,-3.909,-7.147,-5.010,-5.210,-4.001,-5.068,-4.505,-5.018,-0.338
506,2015-01-01 08:26:00-05:00,80.682350,108.606560,167.851850,47.60,126.1,-2.932,393.0,276.8,276.0,...,-3.684,-3.603,-6.987,-4.791,-4.985,-3.954,-4.814,-4.234,-4.825,-0.239
507,2015-01-01 08:27:00-05:00,83.992390,112.412186,173.011350,49.88,130.1,-3.077,395.2,277.0,276.2,...,-3.446,-3.349,-6.876,-4.549,-4.835,-3.840,-4.587,-4.042,-4.633,-0.192
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525075,2015-12-31 15:15:00-05:00,137.189540,124.707275,130.389420,133.40,123.7,9.720,368.6,274.7,274.8,...,12.830,13.310,12.520,12.400,11.910,12.570,-133.600,12.630,11.590,26.300
525076,2015-12-31 15:16:00-05:00,135.889180,123.438736,128.819150,132.30,122.1,9.620,365.8,274.6,274.8,...,12.670,13.120,12.370,12.270,11.800,12.440,-133.500,12.470,11.480,26.270
525077,2015-12-31 15:17:00-05:00,128.086930,117.291180,123.098840,124.90,117.4,9.580,368.8,274.4,274.8,...,12.480,12.920,12.150,12.130,11.680,12.250,-133.600,12.270,11.270,26.200
525078,2015-12-31 15:18:00-05:00,117.092840,107.533180,113.789314,114.30,108.6,9.550,370.8,274.6,274.7,...,12.320,12.760,11.940,12.020,11.560,12.090,-133.900,12.130,11.140,26.190


In [ ]:
pyra_cols = ['Pyra1_Wm2_Avg', 'Pyra2_Wm2_Avg', 'Pyra3_Wm2_Avg']

median_irradiance = canopy_2016[pyra_cols].median(axis=1)

canopy_2016 = canopy_2016[median_irradiance > 100]
canopy_2016

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_1,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10
565,2016-01-01 09:25:00-05:00,101.488340,96.213900,103.358154,103.7,103.9,3.739,371.2,273.5,273.7,...,5.155,5.455,4.896,5.266,4.818,5.389,-137.50,5.411,5.029,9.61
566,2016-01-01 09:26:00-05:00,103.852660,98.653404,105.825740,106.0,106.4,3.728,372.2,274.0,273.4,...,5.183,5.551,4.896,5.280,4.833,5.389,-137.50,5.475,5.090,9.67
567,2016-01-01 09:27:00-05:00,105.271255,100.214680,107.508180,107.4,108.1,3.699,372.8,273.9,273.3,...,5.247,5.600,5.025,5.376,4.914,5.470,-138.00,5.475,5.140,9.70
568,2016-01-01 09:28:00-05:00,105.153040,100.312270,107.620346,107.2,108.2,3.726,374.1,273.8,273.4,...,5.329,5.650,5.007,5.376,4.814,5.485,-137.50,5.492,5.136,9.72
569,2016-01-01 09:29:00-05:00,103.734440,98.848560,106.274390,105.8,106.8,3.742,375.3,273.8,273.4,...,5.344,5.682,4.992,5.376,4.896,5.485,-138.00,5.524,5.169,9.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
526476,2016-12-31 14:36:00-05:00,125.851880,116.777790,124.965440,123.6,122.3,8.100,374.4,276.2,275.0,...,9.390,9.500,9.570,9.680,9.300,9.700,9.87,9.530,9.640,20.86
526477,2016-12-31 14:37:00-05:00,121.137120,112.635780,121.047660,119.4,119.0,8.080,374.4,276.1,275.3,...,9.390,9.510,9.540,9.680,9.270,9.700,9.87,9.560,9.660,20.84
526478,2016-12-31 14:38:00-05:00,115.029366,107.291260,115.747150,113.9,114.3,8.060,374.2,275.9,275.1,...,9.390,9.470,9.530,9.680,9.250,9.700,9.82,9.510,9.590,20.91
526479,2016-12-31 14:39:00-05:00,108.921610,101.679510,110.216170,108.4,109.3,8.030,373.6,276.1,275.2,...,9.250,9.360,9.440,9.620,9.190,9.650,9.76,9.480,9.530,20.87


In [ ]:
canopy_2015.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2015_day.csv', index=False)
canopy_2016.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2016_day.csv', index=False)

#Oprating Mask

In [ ]:
canopy_2015 = pd.read_csv('/content/drive/MyDrive/NIST_processed/canopy_2015_new.csv')
canopy_2016 = pd.read_csv('/content/drive/MyDrive/NIST_processed/canopy_2016_new.csv')

In [ ]:
pyra_cols = ['Pyra1_Wm2_Avg', 'Pyra2_Wm2_Avg', 'Pyra3_Wm2_Avg']

median_irradiance = canopy_2015[pyra_cols].median(axis=1)

canopy_2015['Op_Mask'] = np.where(median_irradiance > 100, 1, 0)
canopy_2015

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10,Op_Mask
0,2015-01-01 00:00:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.229,11.75,277.4,276.5,...,-8.290,-8.680,-8.480,-8.310,-8.660,-8.67,-8.100,-7.542,1.635,0
1,2015-01-01 00:01:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.099,11.69,277.4,276.5,...,-8.350,-8.520,-8.180,-8.020,-8.470,-8.40,-7.988,-7.385,1.635,0
2,2015-01-01 00:02:00-05:00,-4.433096,-8.782201,-4.542599,-0.550,-0.041,-2.185,11.62,277.4,276.5,...,-8.390,-8.470,-8.140,-7.813,-8.380,-8.27,-7.973,-7.366,1.618,0
3,2015-01-01 00:03:00-05:00,-4.433096,-8.977362,-4.542599,-0.547,-0.038,-2.261,11.57,277.4,276.5,...,-8.450,-8.450,-8.100,-7.831,-8.380,-8.25,-7.941,-7.367,1.618,0
4,2015-01-01 00:04:00-05:00,-4.669528,-9.270102,-4.542599,-0.545,-0.036,-2.391,11.49,277.3,276.5,...,-8.450,-8.550,-8.070,-7.799,-8.380,-8.25,-7.973,-7.449,1.539,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525595,2015-12-31 23:55:00-05:00,-3.605585,-7.220922,-3.420970,-0.305,-0.051,5.109,9.64,275.5,275.1,...,3.822,3.646,3.998,3.759,3.845,-138.90,4.077,4.127,9.010,0
525596,2015-12-31 23:56:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.80,275.5,275.2,...,3.808,3.646,3.966,3.742,3.813,-139.10,4.028,4.127,8.980,0
525597,2015-12-31 23:57:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.062,9.85,275.6,275.4,...,3.740,3.628,3.934,3.742,3.749,-138.90,3.999,4.127,8.950,0
525598,2015-12-31 23:58:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.081,9.86,275.7,275.4,...,3.740,3.550,3.934,3.727,3.749,-139.00,3.981,4.110,8.960,0


In [ ]:
pyra_cols = ['Pyra1_Wm2_Avg', 'Pyra2_Wm2_Avg', 'Pyra3_Wm2_Avg']

median_irradiance = canopy_2016[pyra_cols].median(axis=1)

canopy_2016['Op_Mask'] = np.where(median_irradiance > 100, 1, 0)
canopy_2016

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10,Op_Mask
0,2016-01-01 00:00:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.089,9.82,275.6,275.6,...,3.673,3.550,3.838,3.646,3.653,-138.700,3.888,4.095,8.93,0
1,2016-01-01 00:01:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.095,9.60,275.7,275.6,...,3.676,3.550,3.838,3.646,3.653,-138.900,3.867,4.067,8.88,0
2,2016-01-01 00:02:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.098,9.38,275.6,275.6,...,3.644,3.550,3.870,3.646,3.653,-138.300,3.902,4.127,8.86,0
3,2016-01-01 00:03:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.086,9.24,276.0,275.4,...,3.676,3.550,3.934,3.727,3.702,-139.000,3.999,4.127,8.85,0
4,2016-01-01 00:04:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.12,275.7,275.6,...,3.676,3.550,3.934,3.742,3.670,-139.200,3.952,4.127,8.90,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527035,2016-12-31 23:55:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.055,6.882,17.77,276.5,275.1,...,6.237,5.679,6.045,6.018,5.957,6.005,6.047,6.109,9.22,0
527036,2016-12-31 23:56:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.055,6.892,18.85,277.1,275.7,...,6.286,5.646,6.077,6.032,6.004,6.005,6.047,6.094,9.22,0
527037,2016-12-31 23:57:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.059,6.900,19.34,277.3,276.0,...,6.318,5.676,6.059,6.029,6.025,6.005,6.047,6.080,9.22,0
527038,2016-12-31 23:58:00-05:00,-2.946723,-8.952073,-3.168787,-0.305,-0.070,6.901,19.54,277.5,275.9,...,6.315,5.675,6.045,6.032,6.022,6.005,6.047,6.131,9.22,0


In [ ]:
canopy_2015.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2015_new.csv', index=False)
canopy_2016.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2016_new.csv', index=False)

#Feature Engineering

In [ ]:
# Convert the TIMESTAMP column to pandas datetime object
canopy_2015['TIMESTAMP'] = pd.to_datetime(canopy_2015['TIMESTAMP'])

# Set TIMESTAMP as index
canopy_2015.set_index('TIMESTAMP', inplace=True)

###Missing Values and Operating Mask

In [ ]:
# Calculate missing values
missing_counts_2015 = canopy_2015.isna().sum()

# Filter to keep only columns where the count is greater than 0
problem_columns_2015 = missing_counts_2015[missing_counts_2015 > 0]

print(problem_columns_2015.sort_values(ascending=False))

WindSpeedAve_ms          68341
SEWSAmbientTemp_C_Avg    12128
SEWSModuleTemp_C_Avg     12128
SEWSPOAIrrad_Wm2_Avg     12128
RTD_C_Avg_6                366
                         ...  
RTD_C_Avg_5                365
RTD_C_Avg_7                365
RTD_C_Avg_8                365
RTD_C_Avg_9                365
RTD_C_Avg_10               365
Length: 75, dtype: int64


In [ ]:
# Calculate missing values
missing_counts_2016 = canopy_2016.isna().sum()

# Filter to keep only columns where the count is greater than 0
problem_columns_2016 = missing_counts_2016[missing_counts_2016 > 0]

print(problem_columns_2016.sort_values(ascending=False))

WindSpeedAve_ms      2381
RTD_C_Avg_7           930
RTD_C_Avg_6           359
RTD_C_Avg_2           357
RTD_C_Avg_10          357
                     ... 
ShuntPDC_kW_Avg_7     323
ShuntPDC_kW_Avg_3     323
ShuntPDC_kW_Avg_4     323
ShuntPDC_kW_Avg_5     323
ShuntPDC_kW_Avg_6     323
Length: 75, dtype: int64


In [ ]:
canopy_2015.drop(columns=['WindSpeedAve_ms'])
canopy_2016.drop(columns=['WindSpeedAve_ms'])

In [ ]:
pyra_cols = ['Pyra1_Wm2_Avg', 'Pyra2_Wm2_Avg', 'Pyra3_Wm2_Avg']

median_irradiance = canopy_2015[pyra_cols].median(axis=1)

canopy_2015['Op_Mask'] = np.where(median_irradiance > 100, 1, 0)
canopy_2015

,TIMESTAMP,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,...,RTD_C_Avg_2,RTD_C_Avg_3,RTD_C_Avg_4,RTD_C_Avg_5,RTD_C_Avg_6,RTD_C_Avg_7,RTD_C_Avg_8,RTD_C_Avg_9,RTD_C_Avg_10,Op_Mask
0,2015-01-01 00:00:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.229,11.75,277.4,276.5,...,-8.290,-8.680,-8.480,-8.310,-8.660,-8.67,-8.100,-7.542,1.635,0
1,2015-01-01 00:01:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.099,11.69,277.4,276.5,...,-8.350,-8.520,-8.180,-8.020,-8.470,-8.40,-7.988,-7.385,1.635,0
2,2015-01-01 00:02:00-05:00,-4.433096,-8.782201,-4.542599,-0.550,-0.041,-2.185,11.62,277.4,276.5,...,-8.390,-8.470,-8.140,-7.813,-8.380,-8.27,-7.973,-7.366,1.618,0
3,2015-01-01 00:03:00-05:00,-4.433096,-8.977362,-4.542599,-0.547,-0.038,-2.261,11.57,277.4,276.5,...,-8.450,-8.450,-8.100,-7.831,-8.380,-8.25,-7.941,-7.367,1.618,0
4,2015-01-01 00:04:00-05:00,-4.669528,-9.270102,-4.542599,-0.545,-0.036,-2.391,11.49,277.3,276.5,...,-8.450,-8.550,-8.070,-7.799,-8.380,-8.25,-7.973,-7.449,1.539,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525595,2015-12-31 23:55:00-05:00,-3.605585,-7.220922,-3.420970,-0.305,-0.051,5.109,9.64,275.5,275.1,...,3.822,3.646,3.998,3.759,3.845,-138.90,4.077,4.127,9.010,0
525596,2015-12-31 23:56:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.80,275.5,275.2,...,3.808,3.646,3.966,3.742,3.813,-139.10,4.028,4.127,8.980,0
525597,2015-12-31 23:57:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.062,9.85,275.6,275.4,...,3.740,3.628,3.934,3.742,3.749,-138.90,3.999,4.127,8.950,0
525598,2015-12-31 23:58:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.081,9.86,275.7,275.4,...,3.740,3.550,3.934,3.727,3.749,-139.00,3.981,4.110,8.960,0


###Expected vs Actual Power

In [ ]:
P_RATED_KW   = 243.0      # DC nameplate capacity (kW)
GAMMA        = -0.004     # Temperature coefficient for monocrystalline Si (/°C)
T_STC        = 25.0       # Standard Test Condition temperature (°C)
P_AC_MAX_KW  = 225.0      # Inverter AC rated output (~0.93 × DC)

# 1. POA Irradiance
irrad_cols = ['Pyra1_Wm2_Avg', 'Pyra2_Wm2_Avg', 'Pyra3_Wm2_Avg','RefCell1_Wm2_Avg', 'RefCell2_Wm2_Avg']
canopy_2015['POA_fused'] = canopy_2015[irrad_cols].median(axis=1)

# Sensor spread — high spread means sensors disagree (soiling, partial shade, fault)
canopy_2015['irrad_sensor_spread'] = canopy_2015[irrad_cols].max(axis=1) - canopy_2015[irrad_cols].min(axis=1)
canopy_2015['irrad_sensor_cv']     = canopy_2015[irrad_cols].std(axis=1) / canopy_2015[irrad_cols].mean(axis=1).replace(0, np.nan)


# 2. Temperature-Corrected Expected Power
# P_expected = P_rated × (poa_fused / 1000) × [1 + γ × (T_mod - T_STC)]
# This is the power the system should be producing given current irradiance and module temperature.
# Shortfalls indicate a fault signal.

canopy_2015['module_temp'] = canopy_2015['SEWSModuleTemp_C_Avg']   # for readability

canopy_2015['P_expected_kW'] = (P_RATED_KW * (canopy_2015['POA_fused'] / 1000.0) * (1 + GAMMA * (canopy_2015['module_temp'] - T_STC)))

# Expected power cannot be negative (physically impossible)
canopy_2015['P_expected_kW'] = canopy_2015['P_expected_kW'].clip(lower=0)


# 3. Performance Ratio
# PR = actual AC output / expected output
# Healthy system: PR ≈ 0.75–0.85
# PR < 0.70 during clear operation = underperformance
# PR > 1.0 is physically implausible

canopy_2015['PR'] = canopy_2015['InvPAC_kW_Avg'] / canopy_2015['P_expected_kW'].replace(0, np.nan)

# Clip implausible PR values
canopy_2015['PR_clipped'] = canopy_2015['PR'].clip(lower=-0.1, upper=1.2)


# 4. Power Deficit
# Absolute kW gap between expected and actual.
canopy_2015['power_deficit_kW'] = canopy_2015['P_expected_kW'] - canopy_2015['InvPAC_kW_Avg']

# 5. Rolling PR Statistics (temporal context for labeling)
# min_periods avoids NaN at the start of the series taking away too much data
canopy_2015['PR_roll_5min']  = canopy_2015['PR_clipped'].rolling(window=5,  min_periods=3).mean()
canopy_2015['PR_roll_15min'] = canopy_2015['PR_clipped'].rolling(window=15, min_periods=8).mean()
canopy_2015['PR_roll_60min'] = canopy_2015['PR_clipped'].rolling(window=60, min_periods=30).mean()

# Rolling std of PR — captures volatility
canopy_2015['PR_roll_std_15min'] = canopy_2015['PR_clipped'].rolling(window=15, min_periods=8).std()


# 6. Degradation Signal (30-day rolling trend)
# We want to detect PR trending downward over time.
# Method: 30-day rolling mean of PR, then compute % change from the value 30 days before.

# Downsample to hourly for degradation trend (don't need 1-min resolution here)
pr_hourly = canopy_2015['PR_clipped'].resample('1h').mean()

# 30-day rolling mean on hourly series (720 hours = 30 days)
pr_30d_roll = pr_hourly.rolling(window=720, min_periods=360).mean()

# % change vs 30 days ago
pr_30d_pct_change = pr_30d_roll.ffill().pct_change(periods=720) * 100

# Reindex back to 1-min — forward fill within each hour (same degradation rate)
canopy_2015['PR_30d_trend_pct'] = pr_30d_pct_change.reindex(canopy_2015.index, method='ffill')

/tmp/ipykernel_1826/2093012237.py:65: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  pr_30d_pct_change = pr_30d_roll.pct_change(periods=720) * 100


In [ ]:
cols = ['irrad_sensor_spread', 'irrad_sensor_cv', 'P_expected_kW', 'PR', 'PR_clipped',
    'power_deficit_kW', 'PR_roll_5min', 'PR_roll_15min', 'PR_roll_60min', 'PR_roll_std_15min', 'PR_30d_trend_pct']

canopy_2015[cols] = canopy_2015[cols].round(3)

In [ ]:
canopy_2015

,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,InvVc_Avg,...,module_temp,P_expected_kW,PR,PR_clipped,power_deficit_kW,PR_roll_5min,PR_roll_15min,PR_roll_60min,PR_roll_std_15min,PR_30d_trend_pct
TIMESTAMP,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.229,11.75,277.4,276.5,278.1,...,-8.420,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2015-01-01 00:01:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.099,11.69,277.4,276.5,278.2,...,-8.510,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2015-01-01 00:02:00-05:00,-4.433096,-8.782201,-4.542599,-0.550,-0.041,-2.185,11.62,277.4,276.5,278.2,...,-8.530,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2015-01-01 00:03:00-05:00,-4.433096,-8.977362,-4.542599,-0.547,-0.038,-2.261,11.57,277.4,276.5,278.2,...,-8.540,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
2015-01-01 00:04:00-05:00,-4.669528,-9.270102,-4.542599,-0.545,-0.036,-2.391,11.49,277.3,276.5,278.2,...,-8.560,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-31 23:55:00-05:00,-3.605585,-7.220922,-3.420970,-0.305,-0.051,5.109,9.64,275.5,275.1,276.5,...,3.888,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0
2015-12-31 23:56:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.80,275.5,275.2,276.6,...,3.891,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0
2015-12-31 23:57:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.062,9.85,275.6,275.4,276.7,...,3.875,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0


In [ ]:
canopy_2015.to_csv('/content/drive/MyDrive/NIST_processed/canopy_2015_feature_engineered.csv', index=True)

In [ ]:
# Fault Labeling
# Priority within this section: SEVERE > UNDERPERFORMANCE > DEGRADATION > NORMAL

conditions = [
    # Mask: can't evaluate at night or heavy cloud
    canopy_2015['Op_Mask'] == 0,

    # Missing critical inputs
    canopy_2015['P_expected_kW'].isna() | canopy_2015['InvPAC_kW_Avg'].isna(),

    # Severe underperformance: PR < 0.50 sustained over 5-min rolling window
    (canopy_2015['Op_Mask'] == 1) & (canopy_2015['PR_roll_5min'] < 0.50),

    # Underperformance: PR < 0.70 sustained over 10-min rolling window
    (canopy_2015['Op_Mask'] == 1) & (canopy_2015['PR_roll_15min'] < 0.70),
]

choices = [
    'NOT_EVALUABLE',
    'MISSING_DATA',
    'SEVERE_UNDERPERFORMANCE',
    'UNDERPERFORMANCE',
]

canopy_2015['label_power'] = np.select(conditions, choices, default='NORMAL')

# Check results
print("=== Feature Engineering A: Expected vs Actual Power ===")
print(f"P_RATED_KW      : {P_RATED_KW} kW")
print(f"GAMMA           : {GAMMA} /°C  (monocrystalline Si)")
print()
print(f"New columns added:")
new_cols = [
    'POA_fused', 'irrad_sensor_spread', 'irrad_sensor_cv', 'module_temp', 'P_expected_kW', 'PR', 'PR_clipped',
    'power_deficit_kW', 'PR_roll_5min', 'PR_roll_15min', 'PR_roll_60min', 'PR_roll_std_15min', 'PR_30d_trend_pct',
    'label_power']

print(canopy_2015[new_cols].describe().round(3).to_string())

print(f"\nlabel_power distribution:")
print(canopy_2015['label_power'].value_counts())

print(f"\nPR check (Op_Mask=1 rows):")
mask_on = canopy_2015['Op_Mask'] == 1

print(f"  Mean PR (daytime) : {canopy_2015.loc[mask_on, 'PR'].mean():.3f}")
print(f"  Median PR         : {canopy_2015.loc[mask_on, 'PR'].median():.3f}")
print(f"  PR < 0            : {(canopy_2015.loc[mask_on, 'PR'] < 0).sum()} rows")
print(f"  PR > 1.05         : {(canopy_2015.loc[mask_on, 'PR'] > 1.05).sum()} rows")

=== Feature Engineering A: Expected vs Actual Power ===
P_RATED_KW      : 243.0 kW
GAMMA           : -0.004 /°C  (monocrystalline Si)

New columns added:
        POA_fused  irrad_sensor_spread  irrad_sensor_cv  module_temp  P_expected_kW          PR  PR_clipped  power_deficit_kW  PR_roll_5min  PR_roll_15min  PR_roll_60min  PR_roll_std_15min  PR_30d_trend_pct
count  525235.000           525235.000       525235.000   513472.000     513472.000  255007.000  255007.000        513472.000    255111.000     255116.000     255592.000         255116.000        378360.000
mean      163.070               36.057           -0.578       13.823         39.392       0.341       0.788             6.991         0.788          0.789          0.790              0.035             3.972
std       265.009               50.415           99.038       16.079         62.427     145.665       0.277            41.964         0.273          0.267          0.252              0.061            14.974
min        -9.141 

In [ ]:
canopy_2015

,Pyra1_Wm2_Avg,Pyra2_Wm2_Avg,Pyra3_Wm2_Avg,RefCell1_Wm2_Avg,RefCell2_Wm2_Avg,AmbTemp_C_Avg,InvVDVoltage_V_Avg,InvVa_Avg,InvVb_Avg,InvVc_Avg,...,P_expected_kW,PR,PR_clipped,power_deficit_kW,PR_roll_5min,PR_roll_15min,PR_roll_60min,PR_roll_std_15min,PR_30d_trend_pct,label_power
TIMESTAMP,,,,,,,,,,,,,,,,,,,,,
2015-01-01 00:00:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.229,11.75,277.4,276.5,278.1,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NOT_EVALUABLE
2015-01-01 00:01:00-05:00,-4.433096,-8.782201,-4.542599,-0.548,-0.040,-2.099,11.69,277.4,276.5,278.2,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NOT_EVALUABLE
2015-01-01 00:02:00-05:00,-4.433096,-8.782201,-4.542599,-0.550,-0.041,-2.185,11.62,277.4,276.5,278.2,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NOT_EVALUABLE
2015-01-01 00:03:00-05:00,-4.433096,-8.977362,-4.542599,-0.547,-0.038,-2.261,11.57,277.4,276.5,278.2,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NOT_EVALUABLE
2015-01-01 00:04:00-05:00,-4.669528,-9.270102,-4.542599,-0.545,-0.036,-2.391,11.49,277.3,276.5,278.2,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NOT_EVALUABLE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-12-31 23:55:00-05:00,-3.605585,-7.220922,-3.420970,-0.305,-0.051,5.109,9.64,275.5,275.1,276.5,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NOT_EVALUABLE
2015-12-31 23:56:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.080,9.80,275.5,275.2,276.6,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NOT_EVALUABLE
2015-12-31 23:57:00-05:00,-3.605585,-7.123341,-3.420970,-0.305,-0.051,5.062,9.85,275.6,275.4,276.7,...,0.0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NOT_EVALUABLE
